# Mergen V3 - SNN Büyük Türkçe Dil Modeli Eğitim Arayüzü

Bu notebook, 5 TB Google Drive veri kümesindeki Türkçe Wikipedia metin parçalarını (chunks) yüksek performanslı yerel I/O transferi ve senkronize uyku döngüleri (DMN DreamModule) kullanarak SNN (Spiking Neural Network) kortikal kolonu üzerinde eğitmek üzere tasarlanmıştır.

In [ ]:
# 1. Google Drive Bağlantısı
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Donanım Kontrolü
!nvidia-smi

In [ ]:
# 3. Gereksinimlerin Yüklenmesi
!pip install sentence-transformers zeyrek

In [ ]:
# 4. Dizin Hazırlıkları (Google Drive)
import os
CORPUS_DIR = "/content/drive/MyDrive/mergen_data/chunks/"
CHECKPOINT_DIR = "/content/drive/MyDrive/mergen_checkpoints/"

os.makedirs(CORPUS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Veri parçaları dizini: {CORPUS_DIR}")
print(f"Checkpoint yedek dizini: {CHECKPOINT_DIR}")
print("\nLütfen Wikipedia .txt parçalarını (her biri 50-100 MB civarı) CORPUS_DIR dizinine yükleyin.")

## 5. Eğitimi Başlat

Eğitim otonomdur: sıradaki txt parçasını yerel `/content/` diskine indirir, eğitir, VRAM şişmesini önlemek için eğitimi durdurup rüya konsolidasyonunu senkronize çalıştırır ve Drive'a yedekler.

In [ ]:
# Proje klasörüne geçiş (repo Colab'e kopyalanmış veya Drive'dan bağlanmış olmalıdır)
# %cd /content/drive/MyDrive/Mergen-main/

!python scripts/train_colab.py \
    --corpus_dir "/content/drive/MyDrive/mergen_data/chunks/" \
    --checkpoint_dir "/content/drive/MyDrive/mergen_checkpoints/" \
    --mx_path "./mergen_weights.mx" \
    --vocab_path "./mergen_vocab.json" \
    --sleep_interval 500 \
    --sleep_cycles 1000 \
    --som_lr 0.05 \
    --stdp_reward 1.0

## 6. Eğitim İlerleme Durumu & Telemetri

In [ ]:
import json
state_path = "/content/drive/MyDrive/mergen_checkpoints/colab_training_state.json"

if os.path.exists(state_path):
    with open(state_path, 'r') as f:
        data = json.load(f)
    print("=== MERGEN EĞİTİM TELEMETRİSİ ===")
    print(f"Son Güncelleme:          {data.get('last_update')}")
    print(f"Eğitilen Toplam Cümle:    {data.get('total_sentences_trained'):,}")
    print(f"Tamamlanan Parçalar ({len(data.get('processed_chunks', []))}):")
    for chunk in data.get('processed_chunks', []):
        print(f"  - {chunk}")
else:
    print("Eğitim henüz başlamadı veya telemetri dosyası bulunamadı.")